# Day 33: Attributes and Methods

**Today's question:** how can an object have its own *behaviour*, and how can it refuse to get into an invalid state
(like a negative bank balance)?

Before running a cell marked 🤔, predict what it will show.

## 1. From function to method

Yesterday `describe(student)` was a function **outside** the class. A function that only makes sense for one kind of
object belongs **inside** its class. A function inside a class is called a **method**.

In [ ]:
class Student:
    def __init__(self, name, student_id):
        self.name = name
        self.student_id = student_id
        self.marks = []

    def describe(self):
        return f"{self.name} ({self.student_id}) has {len(self.marks)} marks"

🤔 **Predict:** `describe` has one parameter, `self`. How many arguments do we pass when we call it?

In [ ]:
asha = Student("Asha", "S001")
bilal = Student("Bilal", "S002")
bilal.marks.append(64)

print(asha.describe())
print(bilal.describe())

`asha.describe()` is Python's short way of writing `Student.describe(asha)`:

In [ ]:
print(Student.describe(asha))

The object **before the dot** becomes `self`. That is all `self` is: *the specific object the method is working on
right now*. For `bilal.describe()`, `self` is `bilal`.

## 2. Methods that change state

Day 31's dict account, rebuilt as a class. Methods can take extra parameters after `self`.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        self.balance -= amount

In [ ]:
asha = BankAccount("Asha", 100)
asha.deposit(50)
asha.withdraw(30)
print(asha.owner, asha.balance)

| Day 31 (dict + functions) | Day 33 (class + methods) |
|---------------------------|--------------------------|
| `asha = create_account("Asha", 100)` | `asha = BankAccount("Asha", 100)` |
| `deposit(asha, 50)` | `asha.deposit(50)` |
| `asha["balance"]` | `asha.balance` |

The data and its behaviour now travel together.

## 3. Commands and questions

- **Command** methods change state (`deposit`), like `list.append`.
- **Question** methods return information and change nothing, like `str.upper`.

🤔 **Predict:** what does this print?

In [ ]:
print(asha.deposit(50))
print(asha.balance)

`deposit` has no `return`, so the call gives `None` (just like `list.append`). The deposit still happened.

A question method **returns** its answer:

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        self.balance -= amount

    def can_withdraw(self, amount):
        return amount <= self.balance


omar = BankAccount("Omar", 40)
print(omar.can_withdraw(25))
print(omar.can_withdraw(100))

But nothing stops a bad withdrawal yet:

In [ ]:
omar.withdraw(100)
print(omar.balance)

## 4. Validation inside methods

**Check first, change second.** Each method raises `ValueError` *before* touching the state, so a failed call leaves
the object exactly as it was. `__init__` is a method too, so it can check its arguments.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(f"Deposit must be more than 0, got {amount}.")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"Withdrawal must be more than 0, got {amount}.")
        if amount > self.balance:
            raise ValueError(f"Cannot withdraw {amount:.2f}: balance is only {self.balance:.2f}.")
        self.balance -= amount

    def can_withdraw(self, amount):
        return amount > 0 and amount <= self.balance

In [ ]:
omar = BankAccount("Omar", 40)
omar.withdraw(100)

🤔 **Predict:** the withdrawal failed. What is Omar's balance now?

In [ ]:
print(omar.balance)

The method **raises**; the **caller** decides what to do. `except ValueError as error` gives us the exception, and
printing it shows the message we wrote in `raise`:

In [ ]:
for amount in [25, 100, -5]:
    try:
        omar.withdraw(amount)
        print(f"Withdrew {amount}. Balance: {omar.balance}")
    except ValueError as error:
        print("Sorry:", error)

In [ ]:
fatima = BankAccount("Fatima", -20)

Pain point 2 from Day 31 is (almost) solved: the **methods** keep the object valid.
`omar.balance = -500` is still *possible* in Python; our rule is to **change state through methods**: they are the
front door.

## 5. `__str__`: friendly printing

In [ ]:
print(omar)

`__str__` is a special method: Python calls it whenever it needs the object as text (`print`, `str()`, f-strings).
It must **return** a string.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(f"Deposit must be more than 0, got {amount}.")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"Withdrawal must be more than 0, got {amount}.")
        if amount > self.balance:
            raise ValueError(f"Cannot withdraw {amount:.2f}: balance is only {self.balance:.2f}.")
        self.balance -= amount

    def __str__(self):
        return f"{self.owner}'s account: balance {self.balance:.2f}"

In [ ]:
mei = BankAccount("Mei", 250)
mei.deposit(19.5)
print(mei)
print(f"Today: {mei}")
print(str(mei).upper())

## 6. Optional: two objects working together

A method can call other methods of the same object with `self.`, and it can receive **another** object as a
parameter. Every change is also recorded in a `history` list.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(f"Deposit must be more than 0, got {amount}.")
        self.balance += amount
        self.history.append(f"+{amount:.2f}")

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"Withdrawal must be more than 0, got {amount}.")
        if amount > self.balance:
            raise ValueError(f"Cannot withdraw {amount:.2f}: balance is only {self.balance:.2f}.")
        self.balance -= amount
        self.history.append(f"-{amount:.2f}")

    def transfer_to(self, other, amount):
        self.withdraw(amount)      # raises before anything changes if it is not possible
        other.deposit(amount)

    def statement(self):
        lines = [f"Statement for {self.owner}"]
        for entry in self.history:
            lines.append(f"  {entry:>10}")
        lines.append(f"  Balance: {self.balance:.2f}")
        return "\n".join(lines)

    def __str__(self):
        return f"{self.owner}'s account: balance {self.balance:.2f}"

In [ ]:
diego = BankAccount("Diego", 100)
zara = BankAccount("Zara")
diego.transfer_to(zara, 30)
zara.deposit(5)
print(diego.statement())
print(zara.statement())

Inside `transfer_to`, `self` is `diego` and `other` is `zara`: two different objects of the same class.

## 7. Common mistakes (these cells fail on purpose)

In [ ]:
# Mistake 1: forgetting self in a method
class Counter:
    def __init__(self):
        self.count = 0

    def add(amount):
        self.count += amount


Counter().add(5)

In [ ]:
# Mistake 2: forgetting "self." inside a method
class Counter:
    def __init__(self):
        self.count = 0

    def add(self, amount):
        count += amount


Counter().add(5)

Mistake 3: forgetting the brackets. No error, but nothing happens: we only *mentioned* the method.

In [ ]:
tally = Counter()
tally.add

In [ ]:
# Mistake 4: __str__ must RETURN text, not print it
class Counter:
    def __init__(self):
        self.count = 0

    def __str__(self):
        print(f"Count: {self.count}")


print(Counter())

In [ ]:
# Mistake 5: calling a method on the class instead of on an object
BankAccount.deposit(50)

## Recap

1. A method is a function inside a class. `obj.method(x)` means `Class.method(obj, x)`, so `self` is `obj`.
2. **Command** methods change state; **question** methods return information.
3. Methods **check first, change second**: `raise ValueError` keeps the object valid, the caller catches it.
4. `__str__` returns the text that `print(obj)` shows.

**Next time:** objects that contain other objects (a playlist of songs) and testing classes.